In [51]:
import re
import warnings

import pymorphy3
import spacy

nlp = spacy.load("ru_core_news_lg")
morph = pymorphy3.MorphAnalyzer()
warnings.filterwarnings("ignore", category=UserWarning, module="pymorphy3")

COUNTRY_KEYWORDS = {
    "Австралия": ["австралия", "канберра", "сидней", "мельбурн"],
    "Австрия": ["австрия", "вена", "зальцбург"],
    "Азербайджан": ["азербайджан", "баку", "нагорный карабах"],
    "Албания": ["албания", "тирана"],
    "Алжир": ["алжир"],
    "Аргентина": ["аргентина", "буэнос-айрес"],
    "Армения": ["армения", "ереван"],
    "Афганистан": ["афганистан", "кабул", "талибан"],
    "Бангладеш": ["бангладеш", "дакка"],
    "Беларусь": ["беларусь", "белоруссия", "лукашенко"],
    "Бельгия": ["бельгия", "брюссель"],
    "Болгария": ["болгария", "софия"],
    "Боливия": ["боливия"],
    "Босния и Герцеговина": ["босния", "герцеговина"],
    "Бразилия": ["бразилия", "бразилиа", "сан-паулу", "салвадор", "рио-де-жанейро"],
    "Ватикан": ["ватикан", "папа римский"],
    "Великобритания": ["великобритания", "британия", "англия", "лондон"],
    "Венгрия": ["венгрия", "будапешт"],
    "Венесуэла": ["венесуэла"],
    "Гана": ["гана", "аккра"],
    "Гватемала": ["гватемала"],
    "Германия": ["германия", "берлин", "кёльн"],
    "Греция": ["греция", "афины"],
    "Грузия": ["грузия", "тбилиси"],
    "Дания": ["дания", "копенгаген", "датчане"],
    "Зимбабве": ["зимбабве"],
    "Израиль": ["израиль", "тель-авив", "иерусалим"],
    "Индонезия": ["индонезия", "джакарта"],
    "Ирак": ["ирак", "багдад"],
    "Ирландия": ["ирландия", "дублин"],
    "Испания": ["испания", "мадрид", "барселона"],
    "Индия": ["индия", "дели", "мумбаи"],
    "Иран": ["иран", "тегеран"],
    "Италия": ["италия", "рим", "милан", "неаполь", "флоренция", "турин", "венеция"],
    "Йемен": ["йемен", "хуситы"],
    "Камбоджа": ["камбоджа", "пномпень"],
    "Камерун": ["камерун", "яунде"],
    "Канада": ["канада", "оттава", "торонто", "монреаль", "ванкувер"],
    "Казахстан": ["казахстан", "астана", "алматы", "караганда"],
    "Кипр": ["кипр", "никосия"],
    "Киргизия": ["киргизия", "кыргызстан", "бишкек"],
    "Китай": ["китай", "пекин", "си цзиньпин", "кнр"],
    "Коморы": ["коморы", "морони"],
    "Косово": ["косово", "приштина"],
    "Кот-д’Ивуар": ["кот-д'ивуар", "абиджан"],
    "Латвия": ["латвия", "рига"],
    "Ливан": ["ливан", "бейрут"],
    "Литва": ["литва", "вильнюс"],
    "Лихтенштейн": ["лихтенштейн"],
    "Македония": ["македония"],
    "Мальдивы": ["мальдивы"],
    "Мексика": ["мексика", "мехико"],
    "Мозамбик": ["мозамбик", "мапуту"],
    "Молдова": ["молдова", "молдавия", "кишинёв"],
    "Нигерия": ["нигерия", "абуджа"],
    "Нидерланды": ["нидерланды", "голландия", "амстердам", "роттердам", "гаага"],
    "Пакистан": ["пакистан", "исламабад", "каратчи", "лахор", "пенджаб"],
    "Палестина": ["палестина", "хамас", "сектор газа"],
    "Перу": ["перу", "лима"],
    "Польша": ["польша", "варшава"],
    "Португалия": ["португалия", "лиссабон"],
    "Россия": ["россия", "русский", "российская федерация", "рпц", "москва", "московский", "рф",
               "чечня", "чеченский", "санкт-петербург", "путин", "газпром", "telegram",
               "ссср", "хабаровск", "омск", "волгоград", "калининград", "владикавказ", "байкал",
               "астрахань", "дагестан", "сахалин", "курск", "шереметьево"],
    "Румыния": ["румыния", "бухарест"],
    "Сальвадор": ["сальвадор"],
    "Сан-Марино": ["сан-марино", "серравалле"],
    "Северная Корея": ["северная корея", "пхеньян", "кндр"],
    "Сенегал": ["сенегал", "дакар"],
    "Сербия": ["сербия", "белград"],
    "Сингапур": ["сингапур"],
    "Сирия": ["сирия", "дамаск", "алеппо"],
    "Словакия": ["словакия", "братислава"],
    "Словения": ["словения"],
    "Суринам": ["суринам"],
    "США": ["сша", "америка", "вашингтон", "байден", "буш", "обама",
            "microsoft", "aol", "time warner", "apple", "google", "facebook",
            "youtube", "sony", "оон"],
    "Таджикистан": ["таджикистан", "душанбе"],
    "Таиланд": ["таиланд", "бангкок"],
    "Тайвань": ["тайвань", "тайбэй"],
    "Туркменистан": ["туркменистан", "ашхабад"],
    "Турция": ["турция", "эрдоган", "анкара"],
    "Уганда": ["уганда"],
    "Узбекистан": ["узбекистан"],
    "Украина": ["украина", "киев", "зеленский", "донецк", "луганск"],
    "Уругвай": ["уругвай", "монтевидео"],
    "Филиппины": ["филиппины"],
    "Финляндия": ["финляндия", "хельсинки"],
    "Франция": ["франция", "париж", "макрон"],
    "Хорватия": ["хорватия"],
    "Черногория": ["черногория", "подгорица"],
    "Чехия": ["чехия", "прага"],
    "Чили": ["чили", "сантьяго"],
    "Шотландия": ["шотландия", "эдинбург", "глазго"],
    "Швейцария": ["швейцария", "берн"],
    "Швеция": ["швеция", "стокгольм"],
    "Эквадор": ["эквадор", "кито"],
    "Эритрея": ["эритрея", "асмэра", "массауа"],
    "Эстония": ["эстония", "таллин"],
    "Эфиопия": ["эфиопия", "аддис-абеба"],
    "Югославия": ["югославия"],
    "ЮАР": ["юар", "южная африка"],
    "Южная Корея": ["южная корея", "сеул", "пусан"],
    "Япония": ["япония", "токио", "токийский", "хиросима", "осака"],
}

keyword2country = {}
for country, words in COUNTRY_KEYWORDS.items():
    for w in words:
        keyword2country[w.lower()] = country


def find_countries(text: str) -> list[str]:
    if not isinstance(text, str) or not text.strip():
        return []

    doc = nlp(text)
    found = set()
    for token in doc:
        if not token.is_alpha:
            continue
        lemma = morph.parse(token.text)[0].normal_form.lower()
        if lemma in keyword2country:
            found.add(keyword2country[lemma])
    return list(found)


text = "Россия и Китай подписали соглашение о поставках газа в Пекин."
print(find_countries(text))

['Китай', 'Россия']


In [54]:
import pandas as pd

df = pd.read_csv('../events/2_struct/2000-2025.csv')
df["country"] = df["event"].apply(find_countries)

df

,date_start,date_end,event,country
0,2000-01-01,NaN,Деноминация белорусского рубля;,[]
1,2000-01-01,NaN,"В связи с «проблемой-2000», в Иране объявлен н...",[Иран]
2,2000-01-01,NaN,"Вступление в силу закона в Великобритании, сог...",[Великобритания]
3,2000-01-02,NaN,Крушение украинского сухогруза типа «река-море...,[Камбоджа]
4,2000-01-03,NaN,Обстрел из гранатомёта территории российского ...,[Ливан]
...,...,...,...,...
5639,2025-09-18,NaN,На камчатке зафиксировано землетрясение магнит...,[]
5640,2025-09-20,NaN,Проведение конкурса песни «интервидение» в мос...,[Россия]
5641,2025-09-23,NaN,Международный уголовный суд представил подтвер...,[Филиппины]
5642,2025-09-25,NaN,Парламент Кыргызстана объявил о самороспуске.,[Киргизия]


In [55]:
empty_rows = df[df["country"].apply(lambda x: len(x) == 0)]
empty_rows

,date_start,date_end,event,country
0,2000-01-01,NaN,Деноминация белорусского рубля;,[]
15,2000-01-15,NaN,Убийство сербского военного и политического де...,[]
19,2000-01-18,NaN,"В 8:43 жители канадской территории юкон, части...",[]
25,2000-01-24,NaN,В ночь с 25 на 26 января — крушение поезда на ...,[]
29,2000-01-30,NaN,Выброс в Дунай более 100 тыс. кубометров тяжёл...,[]
...,...,...,...,...
5630,2025-09-09,NaN,Антиправительственные акции протеста в Непале....,[]
5632,2025-09-10,NaN,Убийство политического активиста чарли кирка в...,[]
5633,2025-09-11,2025-09-14,турнир по Dota 2 The International 2025.,[]
5638,2025-09-16,NaN,Всеобщие выборы в малави.,[]


In [66]:
nlp = spacy.load('ru_core_news_lg')
nlp.pipe_names

['tok2vec', 'morphologizer', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

In [88]:
def lemmatize(morph, text):
    # 1. Приводим к нижнему регистру
    text = text.lower()

    # 2. Токенизация — оставляем только слова
    tokens = re.findall(r"[а-яА-ЯёЁ]+", text)

    # 3. Лемматизация
    lemmas = [morph.parse(word)[0].normal_form for word in tokens]
    print(lemmas)
    # 4. Возвращаем строку
    return " ".join(lemmas)


# text = 'Tesla is going to aquire twitter for $45 billion'
text = "деноминация белорусского рубля"
morph = pymorphy3.MorphAnalyzer()
doc = nlp(lemmatize(morph, text))

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

['деноминация', 'белорусский', 'рубль']


In [89]:
nlp.pipe_labels['ner']

['LOC', 'ORG', 'PER']